# Health checks
Expose liveness and readiness separately.


In [ ]:
from fastapi import FastAPI
from httpx import ASGITransport, AsyncClient

app = FastAPI()

@app.get("/health")
async def health() -> dict[str, str]:
    return {"status": "ok"}

async with AsyncClient(transport=ASGITransport(app=app), base_url="http://test") as client:
    response = await client.get("/health")
print(response.status_code, response.json())


## Polished version
Liveness checks the process; readiness checks whether it can serve traffic.


In [ ]:
from typing import Protocol
from fastapi import HTTPException

class ReadinessCheck(Protocol):
    async def ready(self) -> bool: ...

class MemoryDatabase:
    def __init__(self, available: bool = True) -> None:
        self.available = available
    async def ready(self) -> bool:
        return self.available

def create_app(database: ReadinessCheck) -> FastAPI:
    service = FastAPI()
    @service.get("/live")
    async def live() -> dict[str, str]:
        return {"status": "alive"}
    @service.get("/ready")
    async def ready() -> dict[str, str]:
        if not await database.ready():
            raise HTTPException(status_code=503, detail="not ready")
        return {"status": "ready"}
    return service

service = create_app(MemoryDatabase())
async with AsyncClient(transport=ASGITransport(app=service), base_url="http://test") as client:
    live_response = await client.get("/live")
    ready_response = await client.get("/ready")
print(live_response.json(), ready_response.json())
